In [1]:
import time
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from dataclasses import dataclass
from sklearn.metrics import average_precision_score

from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
from pathlib import Path

import os
import warnings
warnings.filterwarnings('ignore')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
MODEL_DIR = Path.home() / "models" / "GigaChat3-10B-A1.8B-bf16"
DATA_PATH = '../data/train.csv'
BENCHMARK_PATH = '../data/knowledge_bench_public.csv'
RANDOM_SEED = 42

In [ ]:
import os
from huggingface_hub import snapshot_download


os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

local_dir = "./gigachat_model"

snapshot_download(
    repo_id="ai-sage/GigaChat3-10B-A1.8B-bf16",
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    resume_download=True, 
    max_workers=4  
)



In [6]:
from transformers import DeepseekV3ForCausalLM, AutoTokenizer

model = DeepseekV3ForCausalLM.from_pretrained(
    local_dir,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

Loading weights: 100%|████████████████████████████████████████████████████████████████| 363/363 [00:08<00:00, 42.86it/s]
DeepseekV3ForCausalLM LOAD REPORT from: ./gigachat_model
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.shared_head.head.weight             | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.up_proj.weight   | UNEXPECTED |  | 
model.layers.26.enorm.weight                        | UNEXPECTED |  | 
model.layers.26.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.26.mlp.experts.down_proj               | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.mlp.experts.gate_up_proj            | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_proj.weight | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_

In [7]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import snapshot_download
local_dir = "./gigachat_model"


#MODEL_DIR = "ai-sage/GigaChat3-10B-A1.8B-bf16"  # или другая модель
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ["HF_TOKEN"] = "your_token"

PROBE_LAYERS = [0, 5, 10, 15, 20, 25]

# Загружаем модель
model = AutoModelForCausalLM.from_pretrained(
    local_dir,
    device_map="auto",

    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(local_dir)

# Доступ к слоям
if hasattr(model, 'model') and hasattr(model.model, 'layers'):
    NUM_LAYERS = len(model.model.layers)
else:
    NUM_LAYERS = len(model.layers)

device = next(model.parameters()).device
HIDDEN_SIZE = model.config.hidden_size
PROBE_LAYERS = [l for l in PROBE_LAYERS if l < NUM_LAYERS]

print(f"device: {device}")
print(f"hidden: {HIDDEN_SIZE}")
print(f"layers: {NUM_LAYERS}")
print(f"probe layers: {PROBE_LAYERS}")

device: cuda:0
hidden: 1536
layers: 26
probe layers: [0, 5, 10, 15, 20, 25]


In [8]:
UNCERTAINTY_FEATURES = 19
INTERNAL_FEATURES_PER_LAYER = 6  
INTERNAL_FEATURES = len(PROBE_LAYERS) * INTERNAL_FEATURES_PER_LAYER + 2  

class FeatureAccumulator:

    def __init__(self, model, probe_layers: list[int] = PROBE_LAYERS):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks: list[torch.utils.hooks.RemovableHook] = []
        self._hidden: dict[str, torch.Tensor] = {}

    def attach(self):
        self._hidden.clear()
        for idx in self.probe_layers:
            name = f"layer_{idx}"

            def _make(n):
                def _fn(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return _fn

            self._hooks.append(
                self.model.model.layers[idx].register_forward_hook(_make(name))
            )

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def __enter__(self):
        self.attach()
        return self

    def __exit__(self, *_):
        self.detach()

    def compute_features(
        self,
        logits: torch.Tensor,
        input_ids: torch.Tensor,
        answer_start: int,
    ) -> dict:
        seq_len = input_ids.shape[1]
        n_answer = seq_len - answer_start

        if n_answer == 0:
            return {
                "uncertainty": np.zeros(UNCERTAINTY_FEATURES, dtype=np.float32),
                "internal_scalars": np.zeros(INTERNAL_FEATURES, dtype=np.float32),
                "probe_vec": np.zeros(HIDDEN_SIZE, dtype=np.float32),
            }
        answer_logits = logits[0, answer_start - 1: seq_len - 1, :].float()
        answer_ids = input_ids[0, answer_start:seq_len]
        log_probs = torch.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = torch.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)
        
        probs_sorted, _ = probs.sort(dim=-1, descending=True)
        if probs_sorted.shape[-1] >= 2:
            top1_top2 = (probs_sorted[:, 0] - probs_sorted[:, 1]).mean().item() # new: разрыв между top1 и top2 вероятностями
        else:
            top1_top2 = 0.0

    
        # Коэффициент вариации
        lp_mean = token_lp.mean().item()
        lp_std = token_lp.std().item() if n_answer > 1 else 0.0
        cv = lp_std / (abs(lp_mean) + 1e-6)
        
        # Тренд энтропии (наклон)
        entropy_seq = entropy.cpu().numpy()
        if len(entropy_seq) > 1:
            slope = np.polyfit(range(len(entropy_seq)), entropy_seq, 1)[0]
        else:
            slope = 0.0
            
        # Количество пиков энтропии (>90 перцентиля)
        if len(entropy_seq) > 1:
            p90 = np.percentile(entropy_seq, 90)
            n_spikes = (entropy_seq > p90).sum()
        else:
            n_spikes = 0

        uncertainty_arr = np.array([
            token_lp.mean().item(),
            token_lp.min().item(),
            token_lp.max().item(),
            token_lp.std().item() if n_answer > 1 else 0.0,
            entropy.mean().item(),
            entropy.max().item(),
            entropy.std().item() if n_answer > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(),
            float(n_answer),
            token_lp[0].item(),
            top1.mean().item(),
            top5.mean().item(),
            top1_top2,
            cv,                    # new: коэффициент вариации
            slope,                 # new: тренд энтропии
            float(n_spikes),       # new: количество пиков
            entropy_seq[0] if len(entropy_seq) > 0 else 0.0,  # new: энтропия первого токена
            entropy_seq[-1] if len(entropy_seq) > 0 else 0.0, # new: энтропия последнего токена
            (entropy_seq[-1] - entropy_seq[0]) if len(entropy_seq) > 1 else 0.0,  # new: изменение энтропии
        ], dtype=np.float32)

        # внутри призн
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vector = last_hs[answer_start - 1].cpu().float().numpy()

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[answer_start - 1].norm().item())
            int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).mean().item())
            
            # Добавляем STD норм во время ответа
            if n_answer > 1:
                int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).std().item())
            else:
                int_scalars.append(0.0)

            # Logit Lens
            ans_hs = hs[answer_start - 1: seq_len - 1].unsqueeze(0)
            with torch.no_grad():
                ll = self.model.lm_head(self.model.model.norm(ans_hs)).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            ll_max_prob = ll_p.max(dim=-1).values
            
            int_scalars.append(ll_e.mean().item())
            int_scalars.append(ll_e.std().item())  # new: STD энтропии Logit Lens
            int_scalars.append(ll_max_prob.mean().item())  # new: средняя max вероятность

        # entropy drop / ratio
        # Индексы скорректированы из-за добавленных признаков
        first_e = int_scalars[3]  # первая ll_e.mean (после 3 добавленных признаков на слой)
        last_e = int_scalars[-3]  # последняя ll_e.mean
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal_scalar_arr = np.array(int_scalars, dtype=np.float32)

        self._hidden.clear()
        return {
            "uncertainty": uncertainty_arr,
            "internal_scalars": internal_scalar_arr,
            "probe_vec": probe_vector,
        }


def preprocess(
    tokenizer: AutoTokenizer,
    prompt: str,
    answer: str,
) -> tuple[torch.Tensor, int]:
    """Без изменений, работает с Qwen"""
    messages_prompt = [{"role": "user", "content": prompt}]
    prompt_enc = tokenizer.apply_chat_template(
        messages_prompt,
        add_generation_prompt=True,
        tokenize=True,
    )
    prompt_token_ids = prompt_enc["input_ids"]
    answer_start_idx = len(prompt_token_ids)

    messages_full = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    full_enc = tokenizer.apply_chat_template(messages_full, tokenize=True)
    token_ids = full_enc["input_ids"]
    token_ids = torch.tensor([token_ids], dtype=torch.long)
    
    return token_ids, answer_start_idx

In [9]:
df_train = pd.read_csv("train.csv")

In [10]:
accumulator = FeatureAccumulator(model)
unc_list, int_list, probe_list, label_list = [], [], [], [] 

for _, row in tqdm(df_train.iterrows(), total=len(df_train)):
    token_ids, answer_start_idx = preprocess(tokenizer, row["prompt"], row["model_answer"])
    token_ids = token_ids.to(device)
    
    
    with accumulator:    
        with torch.no_grad():
            outputs = model(token_ids) 

    feats = accumulator.compute_features( 
        outputs.logits,
        token_ids,
        answer_start_idx
    
    )
    
    del outputs

    
    unc_list.append(feats["uncertainty"])
    int_list.append(feats["internal_scalars"])
    probe_list.append(feats["probe_vec"])
    label_list.append(row["is_hallucination"])
    

X_y_train = {
    "uncertainty_X":     np.stack(unc_list).astype(np.float32),
    "internal_scalar_X": np.stack(int_list).astype(np.float32),
    "probe_last_prompt": np.stack(probe_list).astype(np.float32),
    "labels":            np.array(label_list, dtype=np.int32),
} 

100%|███████████████████████████████████████████████████████████████████████████████| 1778/1778 [05:53<00:00,  5.03it/s]


In [12]:
import joblib
joblib.dump(X_y_train, 'X_y_train_giga.joblib')

['X_y_train_giga.joblib']

In [13]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import cross_val_score

class HallucinationClassifier:
    def __init__(self, clf, n_features=50,  use_feature_selection=True):
        # ВSelectKBest отбирает лучшие признаки по взаимной информации
        self.selector = SelectKBest(score_func=mutual_info_classif, k=n_features) if use_feature_selection else None
        self.scaler = StandardScaler()
        
  
        self.clf = clf
        self.threshold = 0.5
    
        self.use_feature_selection = use_feature_selection


    def fit(self, X_y: dict):
        y = X_y["labels"]
        

        X = np.hstack([
            X_y["probe_last_prompt"],     
            X_y["internal_scalar_X"],
            X_y["uncertainty_X"],
        ])
        
        X = self.scaler.fit_transform(X)
        
        
        if self.selector is not None:
            X = self.selector.fit_transform(X, y)
            print(f"После отбора признаков: {X.shape[1]} признаков")
        
     
        cv_scores = cross_val_score(self.clf, X, y, cv=3, scoring='average_precision')
        print(f"CV PR-AUC: {cv_scores.mean():.4f} (+- {cv_scores.std():.4f})")
        
        self.clf.fit(X, y)

    def to_X(self, features: dict) -> np.ndarray:

        X = np.hstack([
            features["probe_vec"].reshape(1, -1),
            features["internal_scalars"].reshape(1, -1),
            features["uncertainty"].reshape(1, -1),
        ])
        X = self.scaler.transform(X)
        
        if self.selector is not None:
            X = self.selector.transform(X)
        
        return X

    def predict_proba(self, features: dict) -> float:
        X = self.to_X(features)
        return float(self.clf.predict_proba(X)[0, 1])

    def predict(self, features: dict) -> tuple[int, float]:
        prob = self.predict_proba(features)
        return int(prob >= self.threshold), prob

In [14]:
from sklearn.linear_model import LogisticRegression
import joblib

best_model = LogisticRegression(
    C=1.0, 
    max_iter=3000, 
    class_weight='balanced', 
    random_state=42
)

clf = HallucinationClassifier(clf=best_model, n_features=50, use_feature_selection=True)
clf.fit(X_y_train)  


joblib.dump(clf, 'classifier_gigachat.pkl')


После отбора признаков: 50 признаков
CV PR-AUC: 0.6879 (+- 0.0138)


['classifier_gigachat.pkl']

In [15]:
@dataclass
class ScoringResult:
    is_hallucination: bool
    is_hallucination_proba: float
    t_model_sec: float = 0.0
    t_overhead_sec: float = 0.0
    t_total_sec: float = 0.0


class GuardianOfTruth:
    def __init__(self, model, tokenizer, classifier: HallucinationClassifier):
        self.model = model
        self.tokenizer = tokenizer
        self.classifier = classifier
        self.accumulator = FeatureAccumulator(model)

    def score(self, prompt: str, answer: str) -> ScoringResult:
        token_ids, answer_start_idx = preprocess(self.tokenizer, prompt, answer)
        device = next(self.model.parameters()).device
        token_ids = token_ids.to(device)

       
       
        t0 = time.perf_counter()
        with self.accumulator:
            with torch.no_grad():
                outputs = self.model(token_ids)
        t_model = time.perf_counter() - t0


        t1 = time.perf_counter()
        features = self.accumulator.compute_features(
            outputs.logits,
            token_ids,
            answer_start_idx,
        )
        del outputs
        
        if features is None:
            return ScoringResult(is_hallucination=False, is_hallucination_proba=0.0)

        label, prob = self.classifier.predict(features)
        t_overhead = time.perf_counter() - t1

        return ScoringResult(
            is_hallucination=bool(label),
            is_hallucination_proba=prob,
            t_model_sec=t_model,
            t_overhead_sec=t_overhead,
            t_total_sec=t_model + t_overhead,
        )

In [16]:
guard = GuardianOfTruth(model, tokenizer, clf)

BENCHMARK_PATH = 'knowledge_bench_public.csv'
df_test = pd.read_csv(BENCHMARK_PATH)

train_results = []
for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="train"):
    res = guard.score(row["prompt"], row["model_answer"])
    train_results.append(res)

df_train_scored = df_train.copy()
df_train_scored["pred_proba"] = [r.is_hallucination_proba for r in train_results]


test_results = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="test"):
    res = guard.score(row["prompt"], row["model_answer"])
    test_results.append(res)

df_test_scored = df_test.copy()
df_test_scored["pred_proba"] = [r.is_hallucination_proba for r in test_results]
df_test_scored["t_model_ms"] = [r.t_model_sec * 1000 for r in test_results]
df_test_scored["t_overhead_ms"] = [r.t_overhead_sec * 1000 for r in test_results]


metrics_df = pd.DataFrame([
    {'split': "Train", 'pr_auc': round(average_precision_score(
        df_train_scored["is_hallucination"],
        df_train_scored["pred_proba"].values), 4)},
    {'split': "Test", 'pr_auc': round(average_precision_score(
        df_test_scored["is_hallucination"],
        df_test_scored["pred_proba"].values), 4)},
])
metrics_df = metrics_df.set_index("split")
print(metrics_df.to_string())


print()
print("Timing (test, per sample):")
print(f"  model forward : {df_test_scored['t_model_ms'].mean():.1f} ms (mean)")
print(f"  overhead      : {df_test_scored['t_overhead_ms'].mean():.1f} ms (mean)")
print(f"  overhead p99  : {df_test_scored['t_overhead_ms'].quantile(0.99):.1f} ms")
overhead_ok = df_test_scored['t_overhead_ms'].quantile(0.99) < 1000
print(f"  {'PASS' if overhead_ok else 'FAIL'}: p99 overhead < 1000 ms")


df_test_scored.to_csv('predictions_gigachat_benchmark_public.csv', index=False)

test: 100%|█████████████████████████████████████████████████████████████████████████| 1044/1044 [03:25<00:00,  5.09it/s]


       pr_auc
split        
Train  0.7135
Test   0.6255

Timing (test, per sample):
  model forward : 173.5 ms (mean)
  overhead      : 13.9 ms (mean)
  overhead p99  : 30.0 ms
  PASS: p99 overhead < 1000 ms


In [ ]:
df_test = pd.read_csv('knowledge_bench_private_no_labels.csv')

df_test = df_test.reset_index(drop=True)


predictions = []
for idx in tqdm(range(len(df_test))):
    row = df_test.iloc[idx]
    res = guard.score(row["prompt"], row["model_answer"])
    predictions.append(res.is_hallucination_proba)


df_test["predict_proba"] = predictions


df_test.to_csv('predictions.csv', index=False)


In [22]:
df_test.head(5)

,prompt,model_answer,predict_proba
0,Чье изображение было помещено на американский ...,Изображение американского президента Авраама Л...,0.039999
1,"Какую актрису изображал плакат, который Энди Д...","В фильме «Побег из Шоушенка», который был снят...",0.040641
2,Какую номинацию получил Мартин Рашент на BRIT ...,Мартин Рашент был номинирован на премию BRIT A...,0.042430
3,Какой термин иногда применяется для обозначени...,Термин «мизандрия» иногда используется для обо...,0.037918
4,Какую сеть (SL policy network) вместо улучшенн...,Авторы AlphaGo на этапе симуляции до конца игр...,0.043552


In [23]:
df_test[['prompt', 'model_answer', 'predict_proba']].to_csv('predictions.csv', index=False)


In [ ]:
df_result = df_test[['prompt', 'model_answer', 'predict_proba']].copy()
df_result.to_csv('preds.csv', index=False)

In [26]:
df_result.sample(5)

,prompt,model_answer,predict_proba
856,Кто в 1304 году осадил восставшего Чахиоглу в ...,В 1304 году осада восставшего Чахиоглу в замке...,0.025161
614,На каком итальянском телеканале был показан ви...,Видеоклип на песню «Shared Creation» группы Ga...,0.043046
469,Какой тип паровоза был указан в предварительно...,В предварительном проекте мощного товарного па...,0.014661
918,"Кем стал Стасандр, назначенный сатрапом Арии и...",Стасандр стал сатрапом Арии и Дрангианы после ...,0.087804
649,Кто в тандеме Ильфа и Петрова выступал с речам...,В тандеме Ильи Арнольдовича Ильфа и Евгения Пе...,0.011610


In [3]:
df_result[['predict_proba']].to_csv('../data/binch/knowledge_bench_private_scores.csv', index=False)